# 06 — Social (publication charts)

Owner reviewed `04-viz` and signed off on the framing (2026-09-25), so this renders
the publication-ready set. Each chart in **two targets** from the same data:
- **Social** — full chrome (title/subtitle/source/watermark), `twitter_landscape`
  1600×900 → `outputs/social/`.
- **Web** — `web_mode=True` (drops title/subtitle/source, keeps the watermark),
  `web` preset 1664×936 → `outputs/web/` (the Pages site supplies its own headings).

**Charts (confirmed with owner):**
1. **Rise of “we”** — self vs collective pronoun rate by decade (SOTU series). The
   headline finding. y-axis label kept; x-axis label dropped (year is obvious).
2. **SOTU — most “I” vs most “we”** — top-10 + bottom-10 presidents by self-share,
   two-tone (self=teal, collective=gold). 19th-c. presidents lean “I”; moderns “we”.
3. **Inaugural — most “I” vs most “we”** — same treatment on the inaugural lens.

Word clouds: the per-president picker (`docs/presidents.html`) is the browse tool;
owner selects which president clouds become individual social posts after reviewing.

**Caveats carried in subtitles / to carry on the page:** curated corpus (major
speeches, not exhaustive); written(≤1912)-vs-spoken SOTU break; ghostwriting (speech
as delivered).

In [ ]:
import sys, os
from pathlib import Path
import duckdb

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

from chart_templates import line_chart, single_ranked_bars
from colors import c
from viz import PRESETS
from IPython.display import display

soc_w, soc_h, _ = PRESETS['twitter_landscape']
web_w, web_h, _ = PRESETS['web']
social_out = PROJECT / 'outputs' / 'social'; social_out.mkdir(parents=True, exist_ok=True)
web_out = PROJECT / 'outputs' / 'web'; web_out.mkdir(parents=True, exist_ok=True)
SRC = 'Miller Center (UVA) speech archive'

con = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))
print('ready:', soc_w, soc_h, '/', web_w, web_h)

## Build the chart tables

From the processed feature tables (`speeches_features`, and the per-president
`chart_sotu_self_share` / `chart_inaug_self_share` built in 04-viz). The
top-10/bottom-10 tables select the 10 highest + 10 lowest self-share presidents and
tag each `grp` (self/coll) so the bars render two-tone.

In [ ]:
import pandas as pd
SELF_C = c('teal')   # leans I / me / my
COLL_C = c('gold')   # leans we / us / our

# Decade line (rise of we) — rebuild here so this notebook is self-contained.
con.execute('DROP TABLE IF EXISTS chart_sotu_by_decade')
con.execute("""CREATE TABLE chart_sotu_by_decade AS
  SELECT CAST(FLOOR(year/10.0)*10 AS INTEGER) AS decade,
         ROUND(AVG(self_per_1k),2) self_1k, ROUND(AVG(collective_per_1k),2) coll_1k
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 ORDER BY decade""")

# per-president self_share tables (rebuild; owned conceptually by 04-viz)
con.execute('DROP TABLE IF EXISTS chart_sotu_self_share')
con.execute("""CREATE TABLE chart_sotu_self_share AS
  SELECT president, ROUND(AVG(self_share),3) self_share FROM speeches_features
  WHERE is_sotu_series GROUP BY 1 HAVING COUNT(*)>=2""")
con.execute('DROP TABLE IF EXISTS chart_inaug_self_share')
con.execute("""CREATE TABLE chart_inaug_self_share AS
  SELECT president, ROUND(AVG(self_share),3) self_share FROM speeches_features
  WHERE speech_type='Inaugural Address' GROUP BY 1""")

def top_bottom(src):
    return con.execute(f"""
      WITH r AS (SELECT president, self_share,
             ROW_NUMBER() OVER (ORDER BY self_share DESC) rt,
             ROW_NUMBER() OVER (ORDER BY self_share ASC) rb FROM {src})
      SELECT president, self_share,
             printf('%.0f%%', self_share*100) AS label,
             CASE WHEN rt<=10 THEN 'self' ELSE 'coll' END AS grp,
             CASE WHEN rt<=10 THEN '{SELF_C}' ELSE '{COLL_C}' END AS color
      FROM r WHERE rt<=10 OR rb<=10 ORDER BY self_share DESC""").df()

sotu_tb = top_bottom('chart_sotu_self_share')
inaug_tb = top_bottom('chart_inaug_self_share')
print('sotu top/bottom:', len(sotu_tb), '| inaug top/bottom:', len(inaug_tb))
sotu_tb.head(3)

## Chart 1 — The rise of “we” (SOTU, self vs collective by decade)

In [ ]:
line_series = [
    {'col': 'coll_1k', 'label': 'collective (we/us/our)', 'color': c('teal')},
    {'col': 'self_1k', 'label': 'self (I/me/my)', 'color': c('spice')},
]
line_df = con.execute('SELECT * FROM chart_sotu_by_decade ORDER BY decade').df()

img = line_chart(df=line_df, x_col='decade', series=line_series,
    x_axis_label='',  # year is obvious — no x title
    y_axis_label='Pronouns per 1,000 words',
    legend=True, label_last=False, y_min=0, x_tick_step=20,
    title='Presidents say “we” far more than they used to',
    subtitle='State of the Union, avg per 1,000 words by decade. Pre-1913 were written messages read by a clerk.',
    source=SRC, img_width=soc_w, img_height=soc_h)
img.save(social_out / '01_rise_of_we.png'); display(img)
imgw = line_chart(df=line_df, x_col='decade', series=line_series,
    x_axis_label='', y_axis_label='Pronouns per 1,000 words',
    legend=True, label_last=False, y_min=0, x_tick_step=20,
    title='', subtitle=None, source=None, web_mode=True,
    img_width=web_w, img_height=web_h)
imgw.save(web_out / '01_rise_of_we.png')

## Chart 2 — State of the Union: most “I” vs most “we” (top 10 / bottom 10)

Two-tone: teal = most self-referential (top 10), gold = most collective (bottom 10).
`sort=None` because the df is already ordered (top block then bottom block).

In [ ]:
legend = [{'label': 'leans “I” (top 10)', 'color': SELF_C},
          {'label': 'leans “we” (bottom 10)', 'color': COLL_C}]
img = single_ranked_bars(df=sotu_tb, category_col='president', value_col='self_share',
    total_label_col='label', color_col='color', legend=legend, sort=None,
    title='State of the Union: the “I” presidents and the “we” presidents',
    subtitle='Share of first-person pronouns that are “I” not “we”. Top 10 & bottom 10 (≥2 SOTUs).',
    source=SRC, img_width=soc_w, img_height=soc_h)
img.save(social_out / '02_sotu_i_vs_we_top_bottom.png'); display(img)
imgw = single_ranked_bars(df=sotu_tb, category_col='president', value_col='self_share',
    total_label_col='label', color_col='color', legend=legend, sort=None,
    title='', subtitle=None, source=None, web_mode=True,
    img_width=web_w, img_height=web_h)
imgw.save(web_out / '02_sotu_i_vs_we_top_bottom.png')

## Chart 3 — Inaugural Address: most “I” vs most “we” (top 10 / bottom 10)

In [ ]:
img = single_ranked_bars(df=inaug_tb, category_col='president', value_col='self_share',
    total_label_col='label', color_col='color', legend=legend, sort=None,
    title='Inaugural addresses: the “I” presidents and the “we” presidents',
    subtitle='Share of first-person pronouns that are “I” not “we”. Top 10 & bottom 10.',
    source=SRC, img_width=soc_w, img_height=soc_h)
img.save(social_out / '03_inaugural_i_vs_we_top_bottom.png'); display(img)
imgw = single_ranked_bars(df=inaug_tb, category_col='president', value_col='self_share',
    total_label_col='label', color_col='color', legend=legend, sort=None,
    title='', subtitle=None, source=None, web_mode=True,
    img_width=web_w, img_height=web_h)
imgw.save(web_out / '03_inaugural_i_vs_we_top_bottom.png')

## Summary & Cleanup

Word clouds are generated by `scripts/build_president_pages.py` (the picker) — owner
selects which president clouds to post after browsing `docs/presidents.html`.

In [ ]:
con.close()
soc = sorted(social_out.glob('*.png')); web = sorted(web_out.glob('*.png'))
print('social:', [p.name for p in soc])
print('web:   ', [p.name for p in web])
print('connection closed')